# Next step: step 5c, 14-value models
**What it adds:** a model is now the **14 parameters that matter**, shape `(14,)`, not SPAM's 22. The last 8
(`epsilon1`, `epsilon2` and the six `rscale` values) do nothing in the extended potential we always run, so
nothing should track or optimize them. `sim` appends fixed values for them (`sim.UNUSED_PARAMS`) right before
it calls SPAM. Why they do nothing: `docs/spam_physics.md`, "Parameters that do nothing".

| | Before | Now |
|---|---|---|
| `sim.PARAM_NAMES` | 22 names | 14 names (`x` … `theta2`) |
| `parse_model_string` | first 22 values | first 14 values (still reads 22- or 34-value strings) |
| `simulate`, `initial_state` | take 22 values | take 14; 22 is an error that points to `parse_model_string` |
| SPAM (`jspam`) | gets 22 | still gets 22: the model, then `sim.UNUSED_PARAMS` |

**How it connects:**
- **Below, in JSPAM:** nothing changes. `jspam` still takes 22 values, exactly as SPAM does.
- **Next, step 6:** `run_orbit` and the timing functions take 14-value models from the start.
- **Step 7b:** the models dataclass holds a batch of models as an `(N, 14)` array, the shape optimizers work in.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**`sim.py`:** `PARAM_NAMES` keeps the first 14 names, and a new constant holds SPAM's last 8:
```python
PARAM_NAMES = (
    "x", "y", "z", "vx", "vy", "vz",
    "mass1", "mass2", "rout1", "rout2",
    "phi1", "phi2", "theta1", "theta2",
)
# epsilon1, epsilon2, rscale1_1..3, rscale2_1..3: never read by the extended potential
UNUSED_PARAMS = (0.3, 0.3, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0)
```
A small helper checks the model and appends them. `simulate` and `initial_state` call it:
```python
def _spam_params(params):
    params = np.asarray(params, dtype=float)
    if params.shape != (len(PARAM_NAMES),):
        raise ValueError("a model is 14 values ... (for a SPAM model string, use parse_model_string)")
    return np.concatenate([params, UNUSED_PARAMS])

return jspam.run_collision(_spam_params(params), seed=seed, **_jspam_args(config))   # was: params
```
`parse_model_string` is unchanged apart from reading `len(PARAM_NAMES)` values, now 14.

**Docs:** `docs/spam_physics.md` (Parameters that do nothing) and `docs/architecture.md` (models are 14 values).

**Tests:** a 14-value model gives exactly the run of the original 22-value line, each of the last 8 changed on
its own leaves a run bit-identical, and 22 values are rejected.

In [7]:
import numpy as np

import jspam  # only to show what SPAM receives; package code reaches SPAM only through sim
from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
config = sim.SimulatorConfig(n1=200, n2=200)

## 1. A model is 14 values
`parse_model_string` reads the first 14 values of the 34-value line:

In [8]:
params = sim.parse_model_string(MODEL_STRING)
print("shape:", params.shape)
dict(zip(sim.PARAM_NAMES, params.tolist()))

shape: (14,)


{'x': -9.93853,
 'y': -4.5805,
 'z': 15.43348,
 'vx': -1.54025,
 'vy': -2.97899,
 'vz': 4.69234,
 'mass1': 34.41671,
 'mass2': 29.49607,
 'rout1': 2.97913,
 'rout2': 4.2501,
 'phi1': 301.0,
 'phi2': 35.5,
 'theta1': 310.80745,
 'theta2': 321.98757}

## 2. What SPAM receives
Before calling SPAM, `sim` appends `sim.UNUSED_PARAMS`, so SPAM gets its usual 22:

In [9]:
spam_22 = np.concatenate([params, sim.UNUSED_PARAMS])   # what sim does internally
print("unused values:", sim.UNUSED_PARAMS)
print("SPAM gets", spam_22.shape, ":", spam_22.tolist())

unused values: (0.3, 0.3, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0)
SPAM gets (22,) : [-9.93853, -4.5805, 15.43348, -1.54025, -2.97899, 4.69234, 34.41671, 29.49607, 2.97913, 4.2501, 301.0, 35.5, 310.80745, 321.98757, 0.3, 0.3, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]


## 3. The last 8 really do nothing
The original line's last 8 are `0.3, 0.3, 0, 0, 0, 0, 0, 0`, different from ours. Run SPAM directly with the
original 22, and through `sim` with our 14: the particles match bit for bit. Then change each of the 8 to an
arbitrary value; every run is still identical.

In [10]:
original_22 = np.array(MODEL_STRING.split(",")[:22], dtype=float)
ours = sim.simulate(params, config, seed=1)
theirs = jspam.run_collision(original_22, seed=1, **sim._jspam_args(config))
print("sim (14 + ours) == SPAM (original 22):", np.array_equal(ours, theirs))

names_8 = ["epsilon1", "epsilon2", "rscale1_1", "rscale1_2", "rscale1_3", "rscale2_1", "rscale2_2", "rscale2_3"]
for i, name in enumerate(names_8, start=14):
    changed = spam_22.copy()
    changed[i] = 7.5
    same = np.array_equal(ours, jspam.run_collision(changed, seed=1, **sim._jspam_args(config)))
    print(f"{name:10s} = 7.5: identical? {same}")

sim (14 + ours) == SPAM (original 22): True
epsilon1   = 7.5: identical? True
epsilon2   = 7.5: identical? True
rscale1_1  = 7.5: identical? True
rscale1_2  = 7.5: identical? True
rscale1_3  = 7.5: identical? True
rscale2_1  = 7.5: identical? True
rscale2_2  = 7.5: identical? True
rscale2_3  = 7.5: identical? True


For contrast, a change to one of the 14, even a small one, moves the particles:

In [11]:
nudged = params.copy()
nudged[sim.PARAM_INDEX["theta1"]] += 1.0   # tilt the primary's disk by 1 degree
moved = sim.simulate(nudged, config, seed=1)
print("identical?", np.array_equal(ours, moved), "  largest shift:", np.abs(moved - ours)[:, :3].max().round(4))

identical? False   largest shift: 0.2092


## 4. 22 values are an error
An old 22-value array is rejected rather than silently cut, with a pointer to `parse_model_string`:

In [12]:
try:
    sim.simulate(original_22, config, seed=1)
except ValueError as error:
    print("ValueError:", error)

ValueError: a model is 14 values, shape (14,), got shape (22,) (for a SPAM model string, use parse_model_string)
